# DATASCAPE 2026 — ResNet-34 Original Labels + Robust Augmentation, 5-Fold

Notebook final untuk satu jalur submission yang sederhana:

- membaca `training_manifest.csv` V1 yang sama dengan baseline CNN;
- memakai **label folder asli**, tanpa output relabeling;
- menggunakan gambar training mentah tanpa crop;
- fine-tune ResNet-34 pretrained pada lima fold;
- memakai augmentasi online LAB-CLAHE, HSV, exposure, blur, JPEG, dan downsampling hanya saat training;
- validation dan test memakai transform tetap: RGB, letterbox, dan normalisasi ImageNet;
- ensemble probabilitas lima fold dan kalibrasi bias kelas berbasis OOF untuk submission.

Aktifkan **T4 GPU**, lalu jalankan **Run all**.


## AI Declaration dan batas pipeline

OpenAI Codex digunakan untuk brainstorming, coding, debugging, dan dokumentasi. ResNet-34 pretrained di-fine-tune hanya pada data resmi. Notebook ini memakai label folder asli dari manifest V1; tidak membaca atau menerapkan hasil pseudo-labeling. Notebook juga tidak membuat crop, tidak menjalankan auxiliary detector, dan tidak menghitung statistik dari test.

CLAHE pada luminance LAB, HSV, blur, exposure, JPEG, dan downsampling digunakan sebagai augmentasi stochastic pada train. Raw image tetap muncul pada sebagian besar batch agar model tidak bergantung pada gambar yang sudah “diperbaiki”.


In [ ]:
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
    'sklearn': 'scikit-learn', 'torch': 'torch',
    'torchvision': 'torchvision', 'tqdm': 'tqdm',
    'kornia': 'kornia',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import kornia.augmentation as K
from kornia.color import hsv_to_rgb, lab_to_rgb, rgb_to_hsv, rgb_to_lab
from kornia.enhance import equalize_clahe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Konfigurasi

`AUTO_BATCH_SIZE=True` sekarang mencoba batch 256 terlebih dahulu dan turun otomatis jika VRAM tidak cukup. `KEEP_RAW_PROBABILITY=0.55` memastikan mayoritas view tetap mendekati gambar asli. Augmentasi fotometrik hanya mensimulasikan variasi yang sudah ditemukan pada training, bukan memaksa semua gambar menjadi kondisi ideal.


In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
INDEX_TO_CLASS = {index: label for label, index in CLASS_TO_INDEX.items()}
COMPETITION = 'datascape-2026-data-mining-competition'

DATA_SOURCE = 'auto'
LOCAL_DATA_PATH = ''
PREPROCESSING_SOURCE = ''
MANIFEST_FILENAME = 'training_manifest.csv'

USE_GOOGLE_DRIVE = True
OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/resnet34_robust_raw_original_labels_5fold_gpuaug')
LOCAL_OUTPUT_DIR = Path('artifacts/resnet34_robust_raw_original_labels_5fold_gpuaug')
EXTRACT_DIR = Path('/content/datascape_dataset')
PREPROCESS_EXTRACT_DIR = Path('/content/datascape_preprocessing_v1')

IMAGE_SIZE = 320
BATCH_SIZE = 256
AUTO_BATCH_SIZE = True
BATCH_CANDIDATES = (256, 224, 192, 160, 128, 96, 64, 48, 32)
MAX_MEMORY_FRACTION = 0.90
MAX_EPOCHS = 20
PATIENCE = 5
WARMUP_EPOCHS = 2
REFERENCE_BATCH_SIZE = 32
REFERENCE_LR = 2e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
BALANCE_STRENGTH = 0.50
NUM_WORKERS = 2
RESUME_FOLDS = True
FOLDS = 5
USE_HORIZONTAL_FLIP_TTA = True
CALIBRATE_CLASS_BIAS = True
CLASS_BIAS_LIMIT = 1.20
MIN_CLASS_BIAS_GAIN = 0.002

KEEP_RAW_PROBABILITY = 0.55
CLAHE_PROBABILITY = 0.30
HSV_PROBABILITY = 0.35
GAMMA_PROBABILITY = 0.25
BLUR_PROBABILITY = 0.12
JPEG_PROBABILITY = 0.15
DOWNSAMPLE_PROBABILITY = 0.12
CLAHE_CLIP_LIMIT = (1.5, 2.0)
HUE_SHIFT = 4
SATURATION_RANGE = (0.82, 1.18)
VALUE_RANGE = (0.78, 1.22)
GAMMA_RANGE = (0.75, 1.35)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read Kaggle credentials from Colab Secrets or the environment.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print('Output:', OUTPUT_DIR)


## 2. Hubungkan manifest V1 dan dataset resmi

Notebook memakai `training_manifest.csv` yang sama dengan baseline CNN. Manifest V1 hanya melakukan pembersihan teknis: file tidak terbaca, exact duplicate berlabel sama, dan exact duplicate lintas label. Kolom `label` tetap berasal dari folder asli; tidak ada pseudo-label atau koreksi label.


In [ ]:
def get_kaggle_token():
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('Set KAGGLE_API_TOKEN di Colab Secrets atau environment.')
    return token


def safe_extract(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_file_root(start, filename):
    start = Path(start).expanduser()
    if start.is_file() and start.name == filename:
        return start.parent
    if start.is_dir() and (start / filename).is_file():
        return start
    if start.is_dir():
        matches = list(start.rglob(filename))
        if matches:
            return matches[0].parent
    raise FileNotFoundError(f'{filename} tidak ditemukan di {start}')


def resolve_preprocessing_root():
    if PREPROCESSING_SOURCE:
        source = Path(PREPROCESSING_SOURCE).expanduser()
        if source.suffix.lower() == '.zip':
            return find_file_root(safe_extract(source, PREPROCESS_EXTRACT_DIR), MANIFEST_FILENAME)
        return find_file_root(source, MANIFEST_FILENAME)
    candidates = [
        Path('/content/datascape_preprocessing'),
        Path('/content/artifacts/preprocessing_colab'),
        Path.cwd() / 'artifacts' / 'preprocessing_colab',
        Path.cwd() / 'artifacts' / 'preprocessing_v1',
    ]
    for candidate in candidates:
        if (candidate / MANIFEST_FILENAME).is_file():
            return candidate
    if IN_COLAB:
        archives = sorted(Path('/content').glob('*preprocess*.zip'))
        if archives:
            return find_file_root(safe_extract(archives[0], PREPROCESS_EXTRACT_DIR), MANIFEST_FILENAME)
    raise FileNotFoundError(
        'Manifest V1 tidak ditemukan. Upload preprocessing_colab.zip ke /content '
        'atau isi PREPROCESSING_SOURCE.'
    )


def find_dataset_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Test_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(f'Root dataset tidak ditemukan di {start}')


def resolve_dataset_root():
    if DATA_SOURCE in {'auto', 'folder'} and LOCAL_DATA_PATH:
        local = Path(LOCAL_DATA_PATH).expanduser()
        if local.exists():
            return find_dataset_root(local)
        if DATA_SOURCE == 'folder':
            raise FileNotFoundError(local)
    for candidate in [Path.cwd() / COMPETITION, Path('/content') / COMPETITION]:
        if DATA_SOURCE == 'auto' and candidate.exists():
            return find_dataset_root(candidate)
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(get_kaggle_token())
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, EXTRACT_DIR))
    return find_dataset_root(downloaded)


PREPROCESS_ROOT = resolve_preprocessing_root()
DATA_ROOT = resolve_dataset_root()
print('Preprocessing V1 root:', PREPROCESS_ROOT)
print('Dataset root:', DATA_ROOT)


In [ ]:
train_frame = pd.read_csv(PREPROCESS_ROOT / MANIFEST_FILENAME)
required = {'filepath', 'label', 'fold', 'split_group', 'pixel_sha256'}
missing = required - set(train_frame.columns)
if missing:
    raise ValueError(f'Kolom manifest V1 tidak lengkap: {sorted(missing)}')

# Tolak manifest hasil relabeling agar eksperimen ini benar-benar memakai label asli.
relabel_columns = {'label_final', 'label_decision', 'oof_predicted_label'}
found_relabel_columns = relabel_columns & set(train_frame.columns)
if found_relabel_columns:
    raise ValueError(
        'Manifest mengandung hasil relabeling dan ditolak: '
        f'{sorted(found_relabel_columns)}. Gunakan training_manifest.csv V1.'
    )

train_frame = train_frame.reset_index(drop=True)
train_frame['image_path'] = train_frame.filepath.map(lambda value: str(DATA_ROOT / Path(value)))
missing_paths = train_frame.loc[~train_frame.image_path.map(lambda value: Path(value).is_file()), 'filepath']
if len(missing_paths):
    raise FileNotFoundError(f'{len(missing_paths)} gambar train tidak ditemukan; contoh: {missing_paths.iloc[0]}')
if train_frame.pixel_sha256.duplicated().any():
    raise ValueError('Exact duplicate masih ada di manifest V1.')
if train_frame.groupby('split_group').fold.nunique().max() != 1:
    raise ValueError('Grup near-duplicate menyeberangi fold.')
if set(train_frame.label) != set(CLASSES):
    raise ValueError('Kelas manifest tidak sesuai.')

template = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype={'filename': str})
test_root = DATA_ROOT / 'Test_Set'
by_name = {path.name: path for path in test_root.rglob('*') if path.is_file()}
missing_test = [name for name in template.filename.astype(str) if name not in by_name]
if missing_test:
    raise FileNotFoundError(f'{len(missing_test)} file test tidak ditemukan; contoh: {missing_test[0]}')
test_frame = pd.DataFrame({
    'filename': template.filename.astype(str),
    'image_path': [str(by_name[name]) for name in template.filename.astype(str)],
})

display(pd.crosstab(train_frame.fold, train_frame.label).reindex(columns=CLASSES, fill_value=0))
print('Train dengan label asli:', len(train_frame), '| Test:', len(test_frame))
print('Class counts:')
display(train_frame.label.value_counts().reindex(CLASSES, fill_value=0).to_frame())


## 3. Raw-image dataset dan augmentasi robust

Tidak ada crop. CPU hanya decode, koreksi orientasi EXIF, letterbox, lalu mengirim tensor uint8 ke GPU. CLAHE, HSV/color jitter, gamma, blur, JPEG, downsample, dan augmentasi spasial dijalankan sekaligus per batch di GPU agar data loader tidak menahan utilisasi GPU. Jalur keep_raw tetap mempertahankan 55% gambar tanpa augmentasi fotometrik.


In [ ]:
class Letterbox:
    def __init__(self, size):
        self.size = int(size)
        self.fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), Image.Resampling.BILINEAR,
                            color=self.fill, centering=(0.5, 0.5))


def load_rgb_image(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            background = Image.new('RGBA', rgba.size, tuple(int(v * 255) for v in IMAGENET_MEAN) + (255,))
            image = Image.alpha_composite(background, rgba).convert('RGB')
        else:
            image = image.convert('RGB')
        return image.copy()


class ToUint8Tensor:
    def __call__(self, image):
        array = np.asarray(image, dtype=np.uint8).copy()
        return torch.from_numpy(array).permute(2, 0, 1)


CPU_LETTERBOX = Letterbox(IMAGE_SIZE)
CPU_TO_TENSOR = ToUint8Tensor()


def cpu_transform(image):
    return CPU_TO_TENSOR(CPU_LETTERBOX(image))


class PackageDataset(Dataset):
    def __init__(self, frame, training=False, with_labels=True):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        self.with_labels = with_labels
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        tensor = cpu_transform(load_rgb_image(row.image_path))
        if self.with_labels:
            return tensor, CLASS_TO_INDEX[row.label], index
        return tensor, index


def balanced_sampler(frame):
    counts = frame.label.value_counts()
    weights = frame.label.map(lambda label: counts[label] ** -BALANCE_STRENGTH).to_numpy(copy=True)
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(frame), replacement=True)


def make_loader(dataset, batch_size, sampler=None):
    options = dict(dataset=dataset, batch_size=batch_size, sampler=sampler, shuffle=False,
                   num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)

preview = PackageDataset(train_frame.head(4), training=True)
print('Preview shape/dtype:', tuple(preview[0][0].shape), preview[0][0].dtype)


## 4. ResNet-34 pretrained dan training lima fold

Semua layer di-fine-tune. Checkpoint dipilih dengan validation macro-F1. Weighted sampling membantu kelas `Lainnya` tanpa mengubah metrik tujuan.


In [ ]:
# --- Fix: PyTorch >= 2.6 Uniform Long dtype ---
import torch.distributions.uniform as _uniform_mod
def _safe_rsample(self, sample_shape=torch.Size()):
    shape = self._extended_shape(sample_shape)
    low = self.low.float() if isinstance(self.low, torch.Tensor) and not self.low.is_floating_point() else self.low
    high = self.high.float() if isinstance(self.high, torch.Tensor) and not self.high.is_floating_point() else self.high
    rand = torch.rand(shape, dtype=low.dtype, device=low.device)
    return low + rand * (high - low)
_uniform_mod.Uniform.rsample = _safe_rsample
# --- End fix ---

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Aktifkan GPU Colab.')

GPU_SPATIAL = K.AugmentationSequential(
    K.RandomHorizontalFlip(p=0.5),
    K.RandomAffine(degrees=5.0, padding_mode='border', p=0.5),
    data_keys=['input'],
).to(DEVICE)

GPU_PHOTOMETRIC = K.AugmentationSequential(
    K.RandomGamma(gamma=GAMMA_RANGE, gain=(1.0, 1.0), p=GAMMA_PROBABILITY),
    K.RandomGaussianBlur(kernel_size=(5, 5), sigma=(0.3, 1.1), p=BLUR_PROBABILITY),
    K.RandomJPEG(jpeg_quality=(45.0, 85.0), p=JPEG_PROBABILITY),
    data_keys=['input'],
).to(DEVICE)

GPU_RANDOM_ERASING = K.RandomErasing(
    scale=(0.02, 0.07), ratio=(0.3, 3.3), value=0.0, p=0.10
).to(DEVICE)
NORMALIZE_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
NORMALIZE_STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


def to_gpu_float(images):
    return images.to(
        DEVICE, dtype=torch.float32, memory_format=torch.channels_last, non_blocking=True
    ).div_(255.0)


def normalize_batch(images):
    return ((images - NORMALIZE_MEAN) / NORMALIZE_STD).contiguous(memory_format=torch.channels_last)


def random_lab_clahe(images):
    apply_mask = torch.rand(images.shape[0], device=DEVICE) < CLAHE_PROBABILITY
    if not apply_mask.any():
        return images
    selected = images[apply_mask]
    lab = rgb_to_lab(selected)
    luminance = (lab[:, :1] / 100.0).clamp(0.0, 1.0)
    clip_limit = float(torch.empty((), device=DEVICE).uniform_(*CLAHE_CLIP_LIMIT))
    corrected = equalize_clahe(luminance, clip_limit=clip_limit, grid_size=(8, 8))
    corrected_lab = torch.cat((corrected * 100.0, lab[:, 1:]), dim=1)
    result = images.clone()
    result[apply_mask] = lab_to_rgb(corrected_lab).clamp(0.0, 1.0)
    return result


def random_hsv(images):
    apply_mask = (torch.rand(images.shape[0], 1, 1, 1, device=DEVICE) < HSV_PROBABILITY)
    hsv = rgb_to_hsv(images)
    hue = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(
        -HUE_SHIFT * math.pi / 180.0, HUE_SHIFT * math.pi / 180.0
    )
    saturation = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(*SATURATION_RANGE)
    value = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(*VALUE_RANGE)
    adjusted = torch.cat((
        torch.remainder(hsv[:, :1] + hue, 2.0 * math.pi),
        (hsv[:, 1:2] * saturation).clamp(0.0, 1.0),
        (hsv[:, 2:3] * value).clamp(0.0, 1.0),
    ), dim=1)
    return torch.where(apply_mask, hsv_to_rgb(adjusted), images)


def random_downsample(images):
    scale = float(torch.empty((), device=DEVICE).uniform_(0.45, 0.80))
    height = max(32, int(images.shape[-2] * scale))
    width = max(32, int(images.shape[-1] * scale))
    degraded = F.interpolate(images, size=(height, width), mode='bilinear', align_corners=False)
    degraded = F.interpolate(degraded, size=images.shape[-2:], mode='bilinear', align_corners=False)
    mask = torch.rand(images.shape[0], 1, 1, 1, device=DEVICE) < DOWNSAMPLE_PROBABILITY
    return torch.where(mask, degraded, images)


def prepare_train_batch(images):
    spatial = GPU_SPATIAL(to_gpu_float(images))
    use_augmented = torch.rand(spatial.shape[0], device=DEVICE) >= KEEP_RAW_PROBABILITY
    images = spatial.clone()
    if use_augmented.any():
        augmented = random_lab_clahe(spatial[use_augmented])
        augmented = random_hsv(augmented)
        augmented = GPU_PHOTOMETRIC(augmented)
        images[use_augmented] = random_downsample(augmented)
    return normalize_batch(GPU_RANDOM_ERASING(images))


def prepare_eval_batch(images):
    return normalize_batch(to_gpu_float(images))


def build_model():
    model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(model.fc.in_features, len(CLASSES)))
    return model.to(DEVICE, memory_format=torch.channels_last)


def probe_batch():
    if not AUTO_BATCH_SIZE:
        return BATCH_SIZE
    total = torch.cuda.get_device_properties(0).total_memory
    for batch in BATCH_CANDIDATES:
        model = optimizer = images = prepared = targets = loss = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_model()
            optimizer = torch.optim.AdamW(model.parameters(), lr=REFERENCE_LR)
            images = torch.randint(0, 256, (batch, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.uint8, device=DEVICE)
            prepared = prepare_train_batch(images)
            targets = torch.randint(0, len(CLASSES), (batch,), device=DEVICE)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = nn.functional.cross_entropy(model(prepared), targets)
            loss.backward(); optimizer.step(); torch.cuda.synchronize()
            fraction = torch.cuda.max_memory_reserved() / total
            print(f'batch {batch}: {fraction:.1%} VRAM')
            if fraction <= MAX_MEMORY_FRACTION:
                return batch
        except torch.cuda.OutOfMemoryError:
            print(f'batch {batch}: OOM')
        finally:
            del model, optimizer, images, prepared, targets, loss
            torch.cuda.empty_cache()
    raise RuntimeError('Tidak ada batch size yang muat.')


BATCH_SIZE = probe_batch()
EVAL_BATCH_SIZE = max(8, BATCH_SIZE)
BASE_LR = REFERENCE_LR * math.sqrt(BATCH_SIZE / REFERENCE_BATCH_SIZE)
print('Training batch:', BATCH_SIZE, '| base LR:', BASE_LR)


def learning_rate_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(MAX_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


def predict(model, data_loader, labeled, use_tta=False):
    model.eval(); probabilities=[]; targets_all=[]; indices_all=[]
    with torch.inference_mode():
        for batch in tqdm(data_loader, desc='predict', leave=False):
            if labeled:
                images, targets, indices = batch
                targets_all.append(targets.numpy())
            else:
                images, indices = batch
            images = prepare_eval_batch(images)
            with torch.autocast('cuda', dtype=torch.float16):
                probs = model(images).softmax(1)
                if use_tta:
                    probs = 0.5 * (probs + model(torch.flip(images, dims=[3])).softmax(1))
            probabilities.append(probs.float().cpu().numpy())
            indices_all.append(indices.numpy())
    order = np.argsort(np.concatenate(indices_all))
    result = {'probabilities': np.concatenate(probabilities)[order]}
    if labeled:
        result['targets'] = np.concatenate(targets_all)[order]
    return result


def train_fold(fold):
    fold_dir = OUTPUT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = fold_dir / 'best.pt'
    complete_path = fold_dir / 'complete.json'
    train = train_frame[train_frame.fold.ne(fold)].reset_index(drop=True)
    valid = train_frame[train_frame.fold.eq(fold)].reset_index(drop=True)
    if RESUME_FOLDS and checkpoint_path.is_file() and complete_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model = build_model(); model.load_state_dict(checkpoint['model_state'])
        return model, valid, checkpoint

    train_loader = make_loader(PackageDataset(train, training=True), BATCH_SIZE, balanced_sampler(train))
    valid_loader = make_loader(PackageDataset(valid), EVAL_BATCH_SIZE)
    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, history = -1.0, 0, []

    for epoch in range(MAX_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train(); total_loss = 0.0; total = 0; started = time.time()
        for images, targets, _ in tqdm(train_loader, desc=f'fold {fold} epoch {epoch + 1}', leave=False):
            images = prepare_train_batch(images)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = criterion(model(images), targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += float(loss.item()) * len(images); total += len(images)
        result = predict(model, valid_loader, True)
        score = f1_score(result['targets'], result['probabilities'].argmax(1), average='macro', zero_division=0)
        row = {'epoch': epoch + 1, 'lr': lr, 'train_loss': total_loss / total,
               'valid_macro_f1': score, 'seconds': time.time() - started}
        history.append(row); pd.DataFrame(history).to_csv(fold_dir / 'history.csv', index=False)
        print(f"fold {fold} epoch {epoch + 1}: loss={row['train_loss']:.4f} macro_f1={score:.4f}")
        if score > best + 1e-6:
            best, stale = score, 0
            torch.save({'model_state': model.state_dict(), 'epoch': epoch + 1, 'macro_f1': score}, checkpoint_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    complete_path.write_text(json.dumps({'fold': fold, 'epoch': checkpoint['epoch'], 'macro_f1': checkpoint['macro_f1']}, indent=2))
    return model, valid, checkpoint


## 5. OOF, ensemble test, dan submission

Setiap fold menghasilkan probabilitas validation dan test. Submission menggunakan rata-rata probabilitas lima model, bukan voting label. Bias keputusan antarkelas dipelajari dari OOF dengan cross-fitting antar-fold; label test tidak pernah digunakan untuk kalibrasi.


In [ ]:
oof_probabilities = np.full((len(train_frame), len(CLASSES)), np.nan, dtype=np.float32)
test_probability_sum = np.zeros((len(test_frame), len(CLASSES)), dtype=np.float64)
test_loader = make_loader(PackageDataset(test_frame, with_labels=False), EVAL_BATCH_SIZE)
fold_rows = []

for fold in range(FOLDS):
    model, valid, checkpoint = train_fold(fold)
    valid_loader = make_loader(PackageDataset(valid), EVAL_BATCH_SIZE)
    valid_result = predict(model, valid_loader, True, use_tta=USE_HORIZONTAL_FLIP_TTA)
    global_indices = train_frame.index[train_frame.fold.eq(fold)].to_numpy()
    oof_probabilities[global_indices] = valid_result['probabilities']

    test_result = predict(model, test_loader, False, use_tta=USE_HORIZONTAL_FLIP_TTA)
    test_probability_sum += test_result['probabilities']
    np.save(OUTPUT_DIR / f'fold_{fold}' / 'test_probabilities.npy', test_result['probabilities'])
    fold_rows.append({'fold': fold, 'best_epoch': checkpoint['epoch'], 'macro_f1': checkpoint['macro_f1']})
    del model
    torch.cuda.empty_cache()

assert np.isfinite(oof_probabilities).all()
test_probabilities = (test_probability_sum / FOLDS).astype(np.float32)
np.save(OUTPUT_DIR / 'oof_probabilities.npy', oof_probabilities)
pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)

def apply_class_bias(probabilities, bias):
    scores = np.log(np.clip(probabilities, 1e-8, 1.0)) + np.asarray(bias, dtype=np.float64)
    scores -= scores.max(axis=1, keepdims=True)
    calibrated = np.exp(scores)
    return (calibrated / calibrated.sum(axis=1, keepdims=True)).astype(np.float32)


def optimize_class_bias(probabilities, targets):
    best_bias = np.zeros(len(CLASSES), dtype=np.float64)
    best_score = f1_score(targets, probabilities.argmax(1), average='macro', zero_division=0)
    # Aman menjadi reference (bias=0); hanya dua derajat bebas agar tidak overfit.
    for radius, step in [(CLASS_BIAS_LIMIT, 0.10), (0.12, 0.02)]:
        center = best_bias.copy()
        rusak_values = np.arange(center[1] - radius, center[1] + radius + step / 2, step)
        lainnya_values = np.arange(center[2] - radius, center[2] + radius + step / 2, step)
        for rusak_bias in rusak_values:
            for lainnya_bias in lainnya_values:
                candidate = np.array([0.0, rusak_bias, lainnya_bias])
                predicted = (np.log(np.clip(probabilities, 1e-8, 1.0)) + candidate).argmax(1)
                score = f1_score(targets, predicted, average='macro', zero_division=0)
                if score > best_score + 1e-12 or (
                    abs(score - best_score) <= 1e-12
                    and np.linalg.norm(candidate) < np.linalg.norm(best_bias)
                ):
                    best_score, best_bias = score, candidate
    return best_bias, best_score


targets = train_frame.label.map(CLASS_TO_INDEX).to_numpy()
raw_predictions = oof_probabilities.argmax(1)
raw_macro_f1 = f1_score(targets, raw_predictions, average='macro', zero_division=0)

# Setiap fold dikalibrasi memakai OOF dari empat fold lain. Ini memberi estimasi
# kalibrasi yang jujur dan lima bias independen untuk diringkas secara robust.
calibrated_oof = np.zeros_like(oof_probabilities)
fold_biases = []
calibration_rows = []
for held_out_fold in range(FOLDS):
    calibration_mask = train_frame.fold.ne(held_out_fold).to_numpy()
    held_out_mask = ~calibration_mask
    bias, calibration_score = optimize_class_bias(
        oof_probabilities[calibration_mask], targets[calibration_mask]
    )
    calibrated_oof[held_out_mask] = apply_class_bias(oof_probabilities[held_out_mask], bias)
    held_out_score = f1_score(
        targets[held_out_mask], calibrated_oof[held_out_mask].argmax(1),
        average='macro', zero_division=0,
    )
    fold_biases.append(bias)
    calibration_rows.append({
        'held_out_fold': held_out_fold,
        'calibration_macro_f1': calibration_score,
        'held_out_macro_f1': held_out_score,
        **{f'bias_{label}': float(bias[index]) for index, label in enumerate(CLASSES)},
    })

crossfit_calibrated_predictions = calibrated_oof.argmax(1)
crossfit_calibrated_macro_f1 = f1_score(
    targets, crossfit_calibrated_predictions, average='macro', zero_division=0
)
calibration_accepted = (
    CALIBRATE_CLASS_BIAS
    and crossfit_calibrated_macro_f1 >= raw_macro_f1 + MIN_CLASS_BIAS_GAIN
)
final_class_bias = (
    np.median(np.stack(fold_biases), axis=0)
    if calibration_accepted else np.zeros(len(CLASSES))
)
calibrated_test_probabilities = apply_class_bias(test_probabilities, final_class_bias)
predictions = crossfit_calibrated_predictions if calibration_accepted else raw_predictions
macro_f1 = crossfit_calibrated_macro_f1 if calibration_accepted else raw_macro_f1

np.save(OUTPUT_DIR / 'oof_probabilities_calibrated.npy', calibrated_oof)
np.save(OUTPUT_DIR / 'test_probabilities_raw.npy', test_probabilities)
np.save(OUTPUT_DIR / 'test_probabilities.npy', calibrated_test_probabilities)
pd.DataFrame(calibration_rows).to_csv(OUTPUT_DIR / 'class_bias_crossfit.csv', index=False)

report = pd.DataFrame(classification_report(targets, predictions, target_names=CLASSES,
                                            output_dict=True, zero_division=0)).T
report.to_csv(OUTPUT_DIR / 'oof_classification_report.csv')
display(pd.DataFrame(calibration_rows))
display(report)
print(f'Raw OOF macro-F1: {raw_macro_f1:.4f} | Cross-fitted calibrated: {crossfit_calibrated_macro_f1:.4f}')
print('Calibration accepted:', calibration_accepted, '| Final class bias:', dict(zip(CLASSES, final_class_bias.round(4))))

matrix = confusion_matrix(targets, predictions, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'OOF macro F1 = {macro_f1:.4f}')
plt.xlabel('Prediksi'); plt.ylabel('Label'); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'oof_confusion_matrix.png', dpi=170); plt.show()

submission = template.copy()
submission['label'] = [INDEX_TO_CLASS[index] for index in calibrated_test_probabilities.argmax(1)]
submission.to_csv(OUTPUT_DIR / 'submission_resnet34_robust_raw_5fold.csv', index=False)
probability_frame = pd.DataFrame(calibrated_test_probabilities, columns=[f'prob_{label}' for label in CLASSES])
probability_frame.insert(0, 'filename', template.filename)
probability_frame.to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)

summary = {
    'model': 'ResNet34 ImageNet pretrained, fully fine-tuned',
    'training_images': int(len(train_frame)),
    'label_source': 'V1 original folder labels',
    'relabeling_used': False,
    'oof_macro_f1_raw': float(raw_macro_f1),
    'oof_macro_f1_crossfit_calibrated': float(crossfit_calibrated_macro_f1),
    'oof_macro_f1_selected': float(macro_f1),
    'class_bias_calibration_requested': CALIBRATE_CLASS_BIAS,
    'class_bias_calibration_applied': bool(calibration_accepted),
    'final_class_bias': {label: float(final_class_bias[index]) for index, label in enumerate(CLASSES)},
    'folds': FOLDS,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'raw_global_only': True,
    'crop_used': False,
    'online_train_augmentations': ['GPU LAB-luminance CLAHE', 'GPU HSV', 'GPU gamma', 'GPU blur', 'GPU JPEG', 'GPU downsample'],
    'augmentation_device': 'cuda',
    'cpu_pipeline': ['decode', 'EXIF transpose', 'RGB', 'letterbox', 'uint8 tensor'],
    'test_transform': ['EXIF transpose', 'RGB', 'letterbox', 'ImageNet normalization'],
    'flip_tta': USE_HORIZONTAL_FLIP_TTA,
    'external_training_images': False,
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)

display(submission.label.value_counts().reindex(CLASSES, fill_value=0).rename('predictions').to_frame())
display(submission.head())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Submission:', OUTPUT_DIR / 'submission_resnet34_robust_raw_5fold.csv')
print('ZIP:', archive_path)
